# Blobsim Run Analysis

Post-hoc analysis of one blobsim Shadow run. Sections are filled in per delivery phase:

1. General network information
2. Per-blob p95 propagation latency
3. Cell possession at the attestation deadline
4. Custody-set fetch-completion heatmaps
5. Per-slot bandwidth consumption

In [ ]:
# Injected by papermill at render time.
run_dir = ""
run_id = ""

In [ ]:
# §0 Setup
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "loaders.py").exists() else Path("notebooks").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from loaders import load_run, region_latency_matrix
from plotly_theme import FLAT_UI_PALETTE, apply_theme, ecdf_trace, percentile_summary

WARMUP_SLOTS = 1  # exclude slot 0 (mesh still forming) from latency stats

run = load_run(run_dir or None)
traffic = run["traffic"]
arrivals = run["arrivals"]
columns = run["columns"]
reconstructions = run["reconstructions"]
pool = run["pool"]
slots = run["slots"]
nodes = run["nodes"]
meta = run["meta"]

pd.options.display.max_rows = 120

## §0 Run summary

In [ ]:
meta_row = meta.iloc[0].to_dict() if not meta.empty else {}
n_events_by_kind = (
    slots["event"].value_counts().to_dict() if not slots.empty else {}
)
summary = pd.DataFrame(
    [
        ("run id", run_id or "(local)"),
        ("output dir", meta_row.get("out_dir")),
        ("seed", meta_row.get("seed")),
        ("slots", meta_row.get("slots")),
        ("validators", meta_row.get("validators")),
        ("blob spammers", meta_row.get("blob_spammers")),
        ("blobs / slot", meta_row.get("blobs_per_slot")),
        ("partial columns", meta_row.get("enable_partial_columns")),
        ("blocks in blobs", meta_row.get("blocks_in_blobs")),
        ("blob reconstruction", meta_row.get("blob_reconstruction_enabled")),
        ("reconstruction trigger", meta_row.get("blob_reconstruction_trigger")),
        ("reconstruction delay (ms)", meta_row.get("blob_reconstruction_delay_ms")),
        ("nodes parsed", len(nodes)),
        ("slot events", n_events_by_kind),
    ],
    columns=["field", "value"],
)
display(summary)

## §1 General network information

Static network shape for this run: how **bandwidth tier**, **region**, and **custody-column count** are distributed across nodes, plus the **inter-region latency** model.

In [ ]:
# §1 Network composition: bandwidth / region / custody / role distributions
cl_nodes = nodes[nodes["is_cl_node"]].copy()

def _bw_key(tier):
    """Sort bandwidth tiers by magnitude (Gbit >> Mbit)."""
    try:
        num, unit = str(tier).split()
        return float(num) * (1000.0 if unit.lower().startswith("g") else 1.0)
    except Exception:
        return 0.0

bw_counts = nodes["bandwidth"].value_counts()
bw_counts = bw_counts.reindex(sorted(bw_counts.index, key=_bw_key, reverse=True))
region_counts = nodes["region"].value_counts().sort_values(ascending=False)
custody_counts = cl_nodes["num_custody_columns"].value_counts().sort_index()
role_counts = nodes["role_group"].value_counts().sort_values(ascending=False)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Bandwidth tier", "Region", "Custody columns (CL nodes)", "Role group"),
    vertical_spacing=0.16, horizontal_spacing=0.12,
)
fig.add_trace(go.Bar(x=bw_counts.index.astype(str), y=bw_counts.values,
                     marker_color=FLAT_UI_PALETTE[0], showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=region_counts.index.astype(str), y=region_counts.values,
                     marker_color=FLAT_UI_PALETTE[1], showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=custody_counts.index.astype(str), y=custody_counts.values,
                     marker_color=FLAT_UI_PALETTE[2], showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(x=role_counts.index.astype(str), y=role_counts.values,
                     marker_color=FLAT_UI_PALETTE[3], showlegend=False), row=2, col=2)
fig.update_yaxes(title_text="nodes")
fig.update_xaxes(title_text="cells / node", row=2, col=1)
fig.update_layout(height=680, title_text=f"Network composition ({len(nodes)} nodes)")
apply_theme(fig)
fig.show(config={"responsive": True})

In [ ]:
# §1 Inter-region latency model (mean over host-to-host topology edges)
mat = region_latency_matrix(run_dir or None)
if mat.empty:
    display(Markdown("_No `topology.gml` found - inter-region latencies unavailable._"))
else:
    fig = go.Figure(go.Heatmap(
        z=mat.to_numpy(dtype=float), x=list(mat.columns), y=list(mat.index),
        colorscale="Viridis", colorbar=dict(title="ms"),
        text=mat.round(0).to_numpy(dtype=float), texttemplate="%{text:.0f}",
        hovertemplate="%{y} -> %{x}: %{z:.0f} ms<extra></extra>",
    ))
    fig.update_layout(height=480, title_text="Mean inter-region one-way latency (ms)")
    apply_theme(fig)
    fig.show(config={"responsive": True})

In [ ]:
# §1 Bandwidth tier x region cross-tab (node counts)
ct = pd.crosstab(nodes["region"], nodes["bandwidth"])
ct = ct.reindex(columns=sorted(ct.columns, key=_bw_key, reverse=True))
ct = ct.reindex(index=nodes["region"].value_counts().index)
fig = go.Figure(go.Heatmap(
    z=ct.to_numpy(dtype=float), x=list(ct.columns.astype(str)), y=list(ct.index.astype(str)),
    colorscale="Blues", colorbar=dict(title="nodes"),
    text=ct.to_numpy(), texttemplate="%{text}",
    hovertemplate="%{y}, %{x}: %{z} nodes<extra></extra>",
))
fig.update_layout(height=440, title_text="Node count by region x bandwidth tier")
apply_theme(fig)
fig.show(config={"responsive": True})

## §2 Per-blob p95 propagation latency

For each blob, latency is measured from its **creation** (the spammer's `blob_offered`) to each node's **first receipt** (provider full-payload or sampler custody-cell arrival). Each bar is one blob, showing the **p95 across receiving nodes**. Only EL-originated (spammer) blobs have a creation timestamp; builder payload-blobs are excluded.

In [ ]:
# §2 Per-blob p95 latency: (first arrival - creation) per node, then p95 across nodes
per_blob = pd.DataFrame()
offered = slots[slots["event"] == "blob_offered"][["blob", "slot", "t_ms"]].copy()
offered = offered.dropna(subset=["blob"]).drop_duplicates("blob")
offered = offered.rename(columns={"t_ms": "creation_t_ms", "slot": "creation_slot"})
arr = arrivals[arrivals["kind"].isin(["full_payload", "custody_cells"])].copy() if not arrivals.empty else arrivals

if offered.empty or arr.empty:
    display(Markdown("_No `blob_offered` / `arrival` events in this run — rebuild the sim (Phase 2 instrumentation) and re-run._"))
else:
    arr["t_ms"] = pd.to_numeric(arr["t_ms"], errors="coerce")
    m = arr.merge(offered, on="blob", how="inner")
    m["latency_ms"] = m["t_ms"] - pd.to_numeric(m["creation_t_ms"], errors="coerce")
    m = m[m["latency_ms"].notna() & (m["latency_ms"] >= 0)]
    # Nodes re-fetch the same blob, so take each node's earliest receipt.
    first = (m.groupby(["blob", "node"], as_index=False)
               .agg(latency_ms=("latency_ms", "min"), creation_slot=("creation_slot", "first")))
    per_blob = (first.groupby("blob")
                .agg(p95_ms=("latency_ms", lambda s: float(np.percentile(s, 95))),
                     p50_ms=("latency_ms", lambda s: float(np.percentile(s, 50))),
                     n_nodes=("latency_ms", "size"),
                     slot=("creation_slot", "first"))
                .reset_index()
                .sort_values("p95_ms")
                .reset_index(drop=True))
    per_blob["rank"] = np.arange(len(per_blob))
    fig = go.Figure(go.Bar(
        x=per_blob["rank"], y=per_blob["p95_ms"] / 1000.0,
        marker=dict(color=per_blob["slot"], colorscale="Viridis", colorbar=dict(title="slot")),
        customdata=np.stack([per_blob["blob"].str.slice(0, 10), per_blob["n_nodes"], per_blob["slot"]], axis=-1),
        hovertemplate="blob %{customdata[0]}..  slot %{customdata[2]}<br>p95 %{y:.2f}s across %{customdata[1]} nodes<extra></extra>",
    ))
    fig.update_layout(height=440,
                      title_text=f"Per-blob p95 propagation latency from creation ({len(per_blob)} blobs)",
                      xaxis_title="blob (ranked by p95)", yaxis_title="p95 latency (s)")
    apply_theme(fig)
    fig.show(config={"responsive": True})

In [ ]:
# §2 Distribution of the per-blob p95 latencies
if per_blob.empty:
    display(Markdown("_No per-blob latencies available._"))
else:
    fig = go.Figure(ecdf_trace(per_blob["p95_ms"] / 1000.0, "per-blob p95",
                               line=dict(color=FLAT_UI_PALETTE[2], width=3)))
    fig.update_xaxes(title_text="per-blob p95 latency (s)")
    fig.update_yaxes(title_text="fraction of blobs", range=[0, 1])
    fig.update_layout(height=380, title_text="ECDF of per-blob p95 latencies")
    apply_theme(fig)
    fig.show(config={"responsive": True})
    display(Markdown(
        f"**{len(per_blob)} blobs** with arrivals &middot; median per-blob p95 "
        f"**{per_blob['p95_ms'].median()/1000:.2f} s** &middot; "
        f"p95-of-p95 **{np.percentile(per_blob['p95_ms'], 95)/1000:.2f} s** &middot; "
        f"max **{per_blob['p95_ms'].max()/1000:.2f} s**"))

## §3 Cell possession at the attestation deadline

For the same 3 sampled slots, first the **beacon-block propagation CDF** — the fraction of CL nodes that had received the block by a given point in the slot (the dashed line marks the t = 4 s attestation deadline; each curve's plateau is the fraction that ever received it). Then, for nodes with **cell-level deltas enabled** that had the block, the fraction of custody cells (`num_custody_columns × cells_per_column`) **already held in CL at the deadline** — derived from the local EL blob pool via getBlobs, before the builder seeds columns over CL.

In [ ]:
# §3 Shared setup + beacon-block propagation CDF for the sampled slots
readiness = slots[slots["event"] == "readiness"].copy() if not slots.empty else pd.DataFrame()
elig = pd.DataFrame()
if not readiness.empty:
    for c in ["slot", "n_blobs", "cells_held", "cells_total"]:
        readiness[c] = pd.to_numeric(readiness[c], errors="coerce")
    readiness["eligible_custody"] = readiness["eligible_custody"].fillna(False).astype(bool)
    elig = readiness[readiness["eligible_custody"]
                     & (readiness["n_blobs"] > 0)
                     & (readiness["cells_total"] > 0)].copy()

block_arr = arrivals[arrivals["kind"] == "block"].copy() if not arrivals.empty else pd.DataFrame()
if not block_arr.empty:
    block_arr["slot"] = pd.to_numeric(block_arr["slot"], errors="coerce")
    block_arr["t_ms"] = pd.to_numeric(block_arr["t_ms"], errors="coerce")
    block_arr["offset_ms"] = block_arr["t_ms"] - block_arr["slot"] * 12_000

# Pick up to 3 slots to feature across §3 (prefer slots with custody-eligible nodes).
slot_pool = (sorted(int(s) for s in elig["slot"].dropna().unique()) if not elig.empty
             else sorted(int(s) for s in block_arr["slot"].dropna().unique()) if not block_arr.empty
             else [])
rng = np.random.default_rng(int(meta_row.get("seed") or 0))
k = min(3, len(slot_pool))
sampled = sorted(int(s) for s in rng.choice(slot_pool, size=k, replace=False)) if k else []

total_cl = int((nodes["is_cl_node"] & ~nodes["is_builder"]).sum()) if not nodes.empty else 0
if block_arr.empty or not sampled:
    display(Markdown("_No beacon-block `arrival` events — rebuild the sim (Phase 3 instrumentation) and re-run._"))
else:
    fig = go.Figure()
    for i, s in enumerate(sampled):
        vals = block_arr[block_arr["slot"] == s]["offset_ms"] / 1000.0
        denom = total_cl if total_cl > 0 else len(vals)
        fig.add_trace(ecdf_trace(vals, f"slot {s} ({len(vals)}/{denom} nodes)", denominator=denom,
                                 line=dict(color=FLAT_UI_PALETTE[i % len(FLAT_UI_PALETTE)], width=3)))
    fig.add_vline(x=4.0, line_dash="dash", line_color=FLAT_UI_PALETTE[2],
                  annotation_text="attestation deadline (4s)", annotation_position="top left")
    fig.update_xaxes(title_text="seconds into slot (beacon-block arrival)")
    fig.update_yaxes(title_text="fraction of CL nodes", range=[0, 1])
    fig.update_layout(height=420, title_text="Beacon-block propagation CDF, by slot")
    apply_theme(fig)
    fig.show(config={"responsive": True})

In [ ]:
# §3 Custody-cell possession at the attestation deadline (t=4s), for the sampled slots
if elig.empty or not sampled:
    display(Markdown("_No `readiness` snapshots with custody-eligible nodes and blobs — rebuild the sim (Phase 3 instrumentation) and re-run._"))
else:
    elig = elig.copy()
    elig["pct_held"] = 100.0 * elig["cells_held"] / elig["cells_total"]
    # Pre-bin with numpy so the top bin [95, 100] INCLUDES exactly-100% possession
    # (plotly's go.Histogram makes the right edge exclusive, which silently drops
    # fully-possessed nodes).
    bin_edges = np.arange(0, 105, 5)
    centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    fig = make_subplots(rows=1, cols=len(sampled), shared_yaxes=True,
                        subplot_titles=[f"slot {s} (n={int((elig['slot'] == s).sum())})" for s in sampled])
    for i, s in enumerate(sampled, start=1):
        vals = elig[elig["slot"] == s]["pct_held"].clip(0, 100).to_numpy()
        counts, _ = np.histogram(vals, bins=bin_edges)
        fig.add_trace(go.Bar(x=centers, y=counts, width=4.6,
                             marker_color=FLAT_UI_PALETTE[i % len(FLAT_UI_PALETTE)],
                             showlegend=False,
                             hovertemplate="%{x:.0f}% held: %{y} nodes<extra></extra>"),
                      row=1, col=i)
        fig.update_xaxes(title_text="% custody cells held", range=[0, 100], row=1, col=i)
    fig.update_yaxes(title_text="nodes", row=1, col=1)
    fig.update_layout(height=400, bargap=0.05,
                      title_text="Custody-cell possession at attestation deadline (t=4s), by slot")
    apply_theme(fig)
    fig.show(config={"responsive": True})
    display(Markdown(
        f"Across custody-eligible node-slots: mean **{elig['pct_held'].mean():.1f}%** &middot; "
        f"median **{elig['pct_held'].median():.1f}%** of custody cells already held in CL after the EL fetch."))

## §4 Custody-set fetch-completion heatmaps

One **heatmap** per **custody-column-count group**: each row is a peer, each column a slot, color is the time (seconds) from **blob release** — the builder seeding the block's columns on CL — until that peer's **entire custody set** finished assembling. `0` = the peer already held its full custody set at release (via getBlobs); **blank** = never completed within the run (censored).

Below the heatmaps, a **completion-time CDF** with one curve per custody-column group: x is time-from-release, y is the fraction of custody fetches completed within time x. The denominator is every attempted fetch (peer × slot), so a curve **plateaus below 1.0** when some fetches never complete.

Finally, an **aggregate completion-time CDF** collapses all custody-column groups into a single curve over every peer, so you can read overall custody-set fetch completion without splitting by group.

In [ ]:
# §4 Custody-set fetch-completion heatmaps (time from blob release), by custody-column group
cc = columns[columns["event"] == "custody_complete"].copy() if not columns.empty else pd.DataFrame()
seeded = slots[slots["event"] == "columns_seeded"][["slot", "t_ms"]].copy() if not slots.empty else pd.DataFrame()

if seeded.empty:
    display(Markdown("_No `columns_seeded` (blob-release) events — rebuild the sim (Phase 4 instrumentation) and re-run._"))
else:
    seeded["slot"] = pd.to_numeric(seeded["slot"], errors="coerce")
    seeded = seeded.rename(columns={"t_ms": "release_t_ms"}).drop_duplicates("slot")
    # Grid = every released slot x every fetcher. Completions fill it in; the rest
    # stay blank (censored). Basing the denominator on releases (not on completions)
    # keeps the count honest even when few/none complete.
    all_slots = sorted(int(s) for s in seeded["slot"].dropna().unique())

    if not cc.empty:
        cc["slot"] = pd.to_numeric(cc["slot"], errors="coerce")
        cc["t_ms"] = pd.to_numeric(cc["t_ms"], errors="coerce")
        cc = (cc.groupby(["node", "slot"], as_index=False)["t_ms"].min()
                .merge(seeded, on="slot", how="left"))
        cc["complete_s"] = ((cc["t_ms"] - cc["release_t_ms"]) / 1000.0).clip(lower=0)  # 0 = ready at release
    else:
        cc = pd.DataFrame(columns=["node", "slot", "complete_s"])

    fetchers = nodes[nodes["is_cl_node"] & ~nodes["is_builder"]][["node", "num_custody_columns"]]
    cc = cc.merge(fetchers, on="node", how="inner")

    completed_cells = 0
    for g in sorted(int(x) for x in fetchers["num_custody_columns"].dropna().unique()):
        grp_nodes = sorted(fetchers[fetchers["num_custody_columns"] == g]["node"].unique())
        sub = cc[cc["num_custody_columns"] == g]
        completed_cells += int(sub[["node", "slot"]].drop_duplicates().shape[0])
        pivot = (sub.pivot_table(index="node", columns="slot", values="complete_s", aggfunc="min")
                 if not sub.empty else pd.DataFrame())
        pivot = pivot.reindex(index=grp_nodes, columns=all_slots)
        fig = go.Figure(go.Heatmap(
            z=pivot.to_numpy(dtype=float), x=[str(s) for s in all_slots], y=list(pivot.index),
            colorscale="Viridis", colorbar=dict(title="s"),
            hovertemplate="peer %{y}  slot %{x}<br>complete +%{z:.2f}s from release<extra></extra>",
        ))
        fig.update_xaxes(title_text="slot", type="category")
        fig.update_yaxes(title_text="peer", showticklabels=(len(grp_nodes) <= 40))
        fig.update_layout(height=max(360, min(1400, 14 * len(grp_nodes))),
                          title_text=f"Custody-set fetch completion from release — {g} custody columns ({len(grp_nodes)} peers)")
        apply_theme(fig)
        fig.show(config={"responsive": True})

    total_expected = len(fetchers) * len(all_slots)
    display(Markdown(
        f"**{completed_cells}/{total_expected}** peer-slots completed their custody set "
        f"(blank heatmap cells = never completed / censored). `0 s` = full set already held at release via getBlobs."))

In [ ]:
# §4 CDF of custody-set fetch-completion times, by custody-column group
# (reuses `cc`, `fetchers`, `all_slots` from the heatmap cell above)
if "complete_s" not in cc.columns or cc.dropna(subset=["complete_s"]).empty:
    display(Markdown("_No custody-set completions to plot (see the heatmap note above)._"))
else:
    fig = go.Figure()
    for i, g in enumerate(sorted(int(x) for x in fetchers["num_custody_columns"].dropna().unique())):
        grp_nodes = fetchers[fetchers["num_custody_columns"] == g]["node"].nunique()
        denom = grp_nodes * len(all_slots)  # all attempted fetches in this group; censored -> plateau < 1
        vals = cc[cc["num_custody_columns"] == g]["complete_s"].dropna()
        if denom == 0 or vals.empty:
            continue
        fig.add_trace(ecdf_trace(
            vals, f"{g} custody cols ({len(vals)}/{denom} = {len(vals)/denom:.0%})",
            denominator=denom,
            line=dict(color=FLAT_UI_PALETTE[i % len(FLAT_UI_PALETTE)], width=3)))
    fig.update_xaxes(title_text="time from blob release (s)")
    fig.update_yaxes(title_text="fraction of custody fetches completed", range=[0, 1])
    fig.update_layout(height=420, title_text="Custody-set fetch-completion CDF, by custody-column group")
    apply_theme(fig)
    fig.show(config={"responsive": True})

In [ ]:
# §4 CDF of custody-set fetch-completion times, all peers (ignoring group)
# (reuses `cc`, `fetchers`, `all_slots` from the heatmap cell above)
if "complete_s" not in cc.columns or cc.dropna(subset=["complete_s"]).empty:
    display(Markdown("_No custody-set completions to plot (see the heatmap note above)._"))
else:
    denom = fetchers["node"].nunique() * len(all_slots)  # every attempted fetch; censored -> plateau < 1
    vals = cc["complete_s"].dropna()
    fig = go.Figure()
    fig.add_trace(ecdf_trace(
        vals, f"all peers ({len(vals)}/{denom} = {len(vals)/denom:.0%})",
        denominator=denom,
        line=dict(color=FLAT_UI_PALETTE[0], width=3)))
    fig.update_xaxes(title_text="time from blob release (s)")
    fig.update_yaxes(title_text="fraction of custody fetches completed", range=[0, 1])
    fig.update_layout(height=420, title_text="Custody-set fetch-completion CDF, all peers")
    apply_theme(fig)
    fig.show(config={"responsive": True})

In [ ]:
# §4 Blob reconstruction latency and outcomes
if reconstructions.empty:
    display(Markdown("_No blob reconstruction events in this run._"))
else:
    starts = reconstructions[
        reconstructions["event"] == "blob_reconstruction_started"
    ].copy()
    terminal = reconstructions[
        reconstructions["event"].isin([
            "blob_reconstruction_completed",
            "blob_reconstruction_dropped",
        ])
    ].copy()
    terminal["outcome_label"] = np.where(
        terminal["event"] == "blob_reconstruction_completed",
        terminal["outcome"].fillna("completed"),
        "dropped:" + terminal["reason"].fillna("unknown").astype(str),
    )
    keys = ["node", "slot", "blob_index", "generation", "attempt", "trigger"]
    joined = starts.merge(
        terminal,
        on=keys,
        how="outer",
        suffixes=("_start", "_terminal"),
    )
    joined["observed_latency_ms"] = (
        pd.to_numeric(joined["t_ms_terminal"], errors="coerce")
        - pd.to_numeric(joined["t_ms_start"], errors="coerce")
    )
    # An attempt with no terminal event is right-censored: the run ended (or the
    # log was cut) before its deadline fired. `outcome` names the terminal event,
    # so `started` is the only count that varies within a group: a queue-capacity
    # drop is rejected before it ever starts.
    joined["outcome"] = joined["outcome_label"].fillna("pending")
    joined["cells_added"] = pd.to_numeric(
        joined["cells_added_terminal"], errors="coerce"
    ).fillna(0)
    summary = (
        joined.groupby(["trigger", "outcome"], dropna=False)
        .agg(
            attempts=("blob_index", "size"),
            started=("t_ms_start", "count"),
            median_latency_ms=("observed_latency_ms", "median"),
            min_latency_ms=("observed_latency_ms", "min"),
            cells_added=("cells_added", "sum"),
        )
        .reset_index()
    )
    display(summary)


## §5 Bandwidth during 3 random host slots

For 3 randomly-sampled (host, slot) pairs, a **stacked cumulative-bandwidth area** — one filled band per **message kind · direction**, the bands adding up (not overlapping) so the top of the stack is the host's total cumulative bytes over the slot. x = slot offset (ms), y = cumulative KiB. One facet per host/slot. Needs the per-message `traffic` events (per-slot `METRIC` aggregates lack intra-slot timing).

In [ ]:
# §5 Bandwidth during 3 random host slots — stacked cumulative area by message kind
import random
import numpy as np
import plotly.express as px

pm = traffic[traffic["event"] == "traffic"].copy() if not traffic.empty else pd.DataFrame()
if pm.empty:
    display(Markdown("_No per-message `traffic` events (only per-slot `METRIC` aggregates, which lack "
                     "intra-slot timing) — rebuild the sim and re-run._"))
else:
    pm["bytes"] = pd.to_numeric(pm["bytes"], errors="coerce").fillna(0.0)
    pm["slot"] = pd.to_numeric(pm["slot"], errors="coerce")
    pm["t_ms"] = pd.to_numeric(pm["t_ms"], errors="coerce")
    pm = pm.dropna(subset=["node", "slot", "t_ms"]).copy()
    pm["slot_i"] = pm["slot"].astype(int)
    pm["slot_offset_ms"] = pm["t_ms"] - pm["slot_i"] * 12_000
    pm = pm[pm["slot_offset_ms"].between(0, 12_000)].copy()

    # Sample 3 (host, slot) pairs from active slots (skip warmup slot 0).
    pool_df = pm[pm["slot_i"] >= WARMUP_SLOTS]
    if pool_df.empty:
        pool_df = pm
    pairs = sorted((str(n), int(s))
                   for n, s in pool_df[["node", "slot_i"]].drop_duplicates().itertuples(index=False))
    selected = sorted(random.Random(str(run_id) or str(meta_row.get("seed"))).sample(pairs, min(3, len(pairs))))
    print(f"Selected host/slot pairs: {selected}")
    pm = pm.merge(pd.DataFrame(selected, columns=["node", "slot_i"]), on=["node", "slot_i"], how="inner")

    if pm.empty:
        display(Markdown("_No bandwidth data for the selected host/slot pairs._"))
    else:
        pm["plot_label"] = pm["node"] + " / slot " + pm["slot_i"].astype(str)
        pm["series"] = pm["kind"].astype(str) + " \u00b7 " + pm["dir"].astype(str)
        pm["offset_bin_ms"] = ((pm["slot_offset_ms"] // 100) * 100).clip(0, 12_000)
        binned = pm.groupby(["plot_label", "series", "offset_bin_ms"], as_index=False)["bytes"].sum()

        # Cumulative per series on a common 100 ms grid (step / forward-filled) so the
        # stacked areas add up cleanly at every x.
        grid = np.arange(0, 12_050, 100)
        rows = []
        for (label, series), g in binned.groupby(["plot_label", "series"]):
            cum = (g.sort_values("offset_bin_ms").set_index("offset_bin_ms")["bytes"].cumsum() / 1024)
            cum = cum.reindex(grid).ffill().fillna(0.0)
            rows.append(pd.DataFrame({"plot_label": label, "series": series,
                                      "offset_bin_ms": grid, "cumulative_kib": cum.values}))
        area = pd.concat(rows, ignore_index=True)

        fig = px.area(
            area, x="offset_bin_ms", y="cumulative_kib", color="series",
            facet_col="plot_label", facet_col_wrap=1,
            color_discrete_sequence=px.colors.qualitative.Dark24,
            title="Bandwidth During 3 Random Host Slots (stacked cumulative by message kind)",
            labels={"offset_bin_ms": "Slot offset (ms)", "cumulative_kib": "Cumulative KiB (stacked)",
                    "series": "Message kind \u00b7 direction", "plot_label": "Host / slot"},
        )
        fig.update_xaxes(range=[0, 12_000])
        fig.update_layout(height=900, legend=dict(orientation="h", y=-0.15))
        fig.show(config={"responsive": True})